# SPAI phase-two training with CvT-13

This notebook initializes SPAI from the **phase-one CvT MFM encoder** and trains the phase-two detector heads with either a fixed or learnable frequency radius. The CvT checkpoint must be passed with `--pretrained`; `--finetune-from` is reserved for a complete phase-two SPAI checkpoint.

Before running, enable a Kaggle GPU and Internet access, then attach or copy a dataset containing `train/0_real`, `train/1_fake`, `val/0_real`, and `val/1_fake`. The default tiny medical dataset and short epoch counts below are a smoke/fine-tuning run, not a replacement for full phase-two training on the paper-scale dataset. To continue an interrupted run, attach its full `ckpt_epoch_*.pth` file and set the matching `*_RESUME_CHECKPOINT` variable below; resuming replaces phase-one initialization and requires the same fixed/learnable-radius configuration. The corresponding `*_EPOCHS` value is the total target epoch count, so it must be greater than the checkpoint's saved epoch.

In [ ]:
# Clone the CvT repository and install Kaggle-safe dependencies.
import os
import subprocess
import sys
from pathlib import Path

os.environ["DISABLE_NEPTUNE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

REPO_URL = "https://github.com/Kashshaf-Labib/spai-parameterized-radius-cvt.git"
REPO_DIR = Path("/kaggle/working/spai-parameterized-radius-cvt")

def run_command(*args):
    command = [str(arg) for arg in args]
    print("RUN:", subprocess.list2cmdline(command))
    subprocess.run(command, check=True)

def run_module(module, *args):
    run_command(sys.executable, "-m", module, *args)

if not REPO_DIR.exists():
    run_command("git", "clone", REPO_URL, REPO_DIR)
run_command("git", "-C", REPO_DIR, "pull", "--ff-only")
os.chdir(REPO_DIR)
run_module("pip", "install", "-q", "-r", "requirements-kaggle.txt")
print("SETUP DONE:", REPO_DIR)

In [ ]:
# Confirm the GPU and the pinned CvT dependency.
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings before training."
assert transformers.__version__ == "5.13.1"

In [ ]:
# Download and strictly validate the phase-one CvT MFM checkpoint.
import gc
import hashlib

from spai.config import get_config
from spai.models.cvt import build_cvt
from spai.utils import extract_mfm_encoder_state_dict

WEIGHT_URL = "https://drive.google.com/file/d/180hbXuIVAkXE3iT8BQO3ivoV1mBjz-r7/view?usp=drive_link"
WEIGHT_PATH = Path("weights/cvt_mfm_pretrain.pth")
EXPECTED_SHA256 = "016a6ae03fcf4297803907d354ad519ecc406867586e2e6e7fac391f32264c33"
WEIGHT_PATH.parent.mkdir(parents=True, exist_ok=True)
if not WEIGHT_PATH.exists():
    run_module("gdown", "--fuzzy", WEIGHT_URL, "-O", WEIGHT_PATH)

digest = hashlib.sha256()
with WEIGHT_PATH.open("rb") as checkpoint_file:
    for chunk in iter(lambda: checkpoint_file.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()
assert actual_sha256 == EXPECTED_SHA256, f"Unexpected checkpoint SHA-256: {actual_sha256}"

checkpoint = torch.load(WEIGHT_PATH, map_location="cpu", weights_only=False)
assert checkpoint["config"].MODEL.TYPE == "cvt"
assert checkpoint["config"].MODEL.CVT.NAME == "cvt_13"
encoder_state = extract_mfm_encoder_state_dict(checkpoint["model"])
assert len(encoder_state) == 455, f"Expected 455 CvT encoder entries, got {len(encoder_state)}"

cvt_config = get_config({"cfg": "configs/spai_cvt.yaml"})
backbone = build_cvt(cvt_config)
backbone.load_state_dict(encoder_state, strict=True)
print(f"VALID CVT MFM CHECKPOINT | epoch={checkpoint.get('epoch')} | entries={len(encoder_state)} | sha256={actual_sha256}")
del checkpoint, encoder_state, backbone
gc.collect()

## Dataset CSV

Edit `DATASET_ROOT` if the dataset is attached under `/kaggle/input/...`. Class directory names must contain `0_real` and `1_fake`, as required by the repository's CSV builder. `CSV_ROOT` is passed consistently to both CSV creation and training so absolute Kaggle input locations work.

In [ ]:
# Build the dataset list used by the smoke/fine-tuning run.
import pandas as pd

DATASET_ROOT = Path("medical_spai_dataset").resolve()  # Change when using /kaggle/input.
TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "val"
CSV_ROOT = DATASET_ROOT.parent
CSV_PATH = Path("datasets/medical_smoke.csv")
assert TRAIN_DIR.is_dir() and VAL_DIR.is_dir(), f"Dataset not found under {DATASET_ROOT}"
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

run_module(
    "spai.tools.create_dir_csv",
    "--train_dir", TRAIN_DIR,
    "--val_dir", VAL_DIR,
    "-o", CSV_PATH,
    "-r", CSV_ROOT,
)

dataset_df = pd.read_csv(CSV_PATH)
print(dataset_df.groupby(["split", "class"]).size())

In [ ]:
# Fixed-radius CvT-SPAI smoke run. Phase-two heads start from random initialization.
from datetime import datetime, timezone

RUN_TAG = datetime.now(timezone.utc).strftime("smoke_%Y%m%dT%H%M%S%fZ")
FIXED_OUTPUT = Path("output/cvt_fixed_radius")
FIXED_EPOCHS = 2  # Use the full schedule in configs/spai_cvt.yaml for real training.
FIXED_RESUME_CHECKPOINT = None  # Or Path("/kaggle/input/.../ckpt_epoch_N.pth")

fixed_train_args = [
    "train",
    "--cfg", "configs/spai_cvt.yaml",
    "--batch-size", 4,
    "--data-path", CSV_PATH,
    "--csv-root-dir", CSV_ROOT,
    "--output", FIXED_OUTPUT,
    "--tag", RUN_TAG,
    "--amp-opt-level", "O0",
    "--data-workers", 2,
    "--save-all",
    "--opt", "TRAIN.EPOCHS", FIXED_EPOCHS,
    "--opt", "TRAIN.WARMUP_EPOCHS", 1,
    "--opt", "PRINT_FREQ", 5,
    "--opt", "MODEL.FEATURE_EXTRACTION_BATCH", 128,
    "--opt", "DATA.TEST_PREFETCH_FACTOR", 1,
]
if FIXED_RESUME_CHECKPOINT is None:
    fixed_train_args.extend(("--pretrained", WEIGHT_PATH))
else:
    fixed_train_args.extend(("--resume", FIXED_RESUME_CHECKPOINT))
run_module("spai", *fixed_train_args)

FIXED_CHECKPOINT_DIR = FIXED_OUTPUT / "finetune_cvt" / RUN_TAG
assert list(FIXED_CHECKPOINT_DIR.glob("ckpt_epoch_*.pth")), FIXED_CHECKPOINT_DIR

In [ ]:
# Learnable-radius CvT-SPAI smoke run, initialized from the same phase-one encoder.
LEARNABLE_OUTPUT = Path("output/cvt_learnable_radius")
LEARNABLE_EPOCHS = 5
LEARNABLE_RESUME_CHECKPOINT = None  # Or Path("/kaggle/input/.../ckpt_epoch_N.pth")

learnable_train_args = [
    "train",
    "--cfg", "configs/spai_cvt_learnable_radius.yaml",
    "--batch-size", 4,
    "--data-path", CSV_PATH,
    "--csv-root-dir", CSV_ROOT,
    "--output", LEARNABLE_OUTPUT,
    "--tag", RUN_TAG,
    "--amp-opt-level", "O0",
    "--data-workers", 2,
    "--save-all",
    "--opt", "TRAIN.EPOCHS", LEARNABLE_EPOCHS,
    "--opt", "TRAIN.WARMUP_EPOCHS", 1,
    "--opt", "TRAIN.RADIUS_LR", 0.05,
    "--opt", "PRINT_FREQ", 5,
    "--opt", "MODEL.FEATURE_EXTRACTION_BATCH", 128,
    "--opt", "DATA.TEST_PREFETCH_FACTOR", 1,
]
if LEARNABLE_RESUME_CHECKPOINT is None:
    learnable_train_args.extend(("--pretrained", WEIGHT_PATH))
else:
    learnable_train_args.extend(("--resume", LEARNABLE_RESUME_CHECKPOINT))
run_module("spai", *learnable_train_args)

LEARNABLE_CHECKPOINT_DIR = LEARNABLE_OUTPUT / "finetune_cvt_learnable_radius" / RUN_TAG
assert list(LEARNABLE_CHECKPOINT_DIR.glob("ckpt_epoch_*.pth")), LEARNABLE_CHECKPOINT_DIR

In [ ]:
# Read the learned radius from each saved phase-two checkpoint.
learnable_checkpoints = sorted(LEARNABLE_CHECKPOINT_DIR.glob("ckpt_epoch_*.pth"))
for checkpoint_path in learnable_checkpoints:
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    radius = state["model"]["mfvit.soft_frequency_mask.radius"].item()
    print(f"{checkpoint_path.name}: learned radius = {radius:.4f}")
    del state

In [ ]:
# Evaluate every saved fixed- and learnable-radius checkpoint on the validation split.
print("========== CVT FIXED RADIUS ==========")
run_module(
    "spai", "test",
    "--cfg", "configs/spai_cvt.yaml",
    "--batch-size", 1,
    "--model", FIXED_CHECKPOINT_DIR,
    "--output", "output/test_cvt_fixed",
    "--tag", f"{RUN_TAG}_fixed",
    "--split", "val",
    "--test-csv", CSV_PATH,
    "--test-csv-root-dir", CSV_ROOT,
    "--opt", "MODEL.PATCH_VIT.MINIMUM_PATCHES", 4,
    "--opt", "DATA.NUM_WORKERS", 2,
    "--opt", "MODEL.FEATURE_EXTRACTION_BATCH", 128,
    "--opt", "DATA.TEST_PREFETCH_FACTOR", 1,
)

print("========== CVT LEARNABLE RADIUS ==========")
run_module(
    "spai", "test",
    "--cfg", "configs/spai_cvt_learnable_radius.yaml",
    "--batch-size", 1,
    "--model", LEARNABLE_CHECKPOINT_DIR,
    "--output", "output/test_cvt_learnable",
    "--tag", f"{RUN_TAG}_learnable",
    "--split", "val",
    "--test-csv", CSV_PATH,
    "--test-csv-root-dir", CSV_ROOT,
    "--opt", "MODEL.PATCH_VIT.MINIMUM_PATCHES", 4,
    "--opt", "DATA.NUM_WORKERS", 2,
    "--opt", "MODEL.FEATURE_EXTRACTION_BATCH", 128,
    "--opt", "DATA.TEST_PREFETCH_FACTOR", 1,
)

In [ ]:
# Package checkpoints, logs, and evaluation results for download from Kaggle.
import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/spai_cvt_results", "gztar", root_dir=REPO_DIR, base_dir="output"
)
print("RESULT ARCHIVE:", archive_path)

## Moving beyond the smoke run

For a meaningful CvT-SPAI detector, point `CSV_PATH` at the complete phase-two training data, remove the short `TRAIN.EPOCHS` overrides, and retain `--pretrained weights/cvt_mfm_pretrain.pth`. Both configurations freeze the phase-one CvT encoder and train the newly initialized phase-two SPAI components; the learnable configuration additionally optimizes the masking radius.